### 开场白

        各位面试官们好。我是来自华东理工大学，信息与工程学院的大三学生。我的专业是机器人工程。我将在接下来的十分多种向你们介绍我完成的agent。

####  项目定位

        我要介绍的是我在今年四月份使用cursor和codex辅助设计完成的完成的机器人听觉感知系统。这个系统实现了让机器人完成从声音输入、身份确认、语义理解，到动作执行和语音反馈的完整闭环。

#### 系统流程

整个系统的主流程是：  

`语音采集 -> 声纹校验 -> ASR -> NLU -> 控制执行 -> TTS反馈`

- **第一步是语音采集**，也就是通过麦克风实时获取用户的音频输入。不仅仅可以通过系统麦克风，也可以使用耳机麦克风进行输入。

- **第二步是声纹检验**。在系统第一次启动时，会自动检测是否有声纹json文件，权重文件，从而录入使用者的声纹信息，确保在机器人控制的时候只有使用者能语音控制系统。这样做既可以提高控制系统的安全性，也可以减少无关人声对系统的干扰。

- **第三步是`ASR`，自动语音识别**。目前我是通过`SpeechRecognition` 调用 Google Web Speech API，把音频转成中文文本。

- **第四步是NLU(natural_language_understanding),自然语言理解**。这里我没有使用大模型进行理解和处理，而是针对机器人的控制场景进行了解析。比如系统能识别“前进”“后退”“左转”，也能进一步抽取“1米”“30度”“慢慢”“快速”这类参数，还能把“前进1m然后左转30度再前进1m”切成多段动作。

- **第五步是控制执行**。解析完成后，系统把命令转成结构化动作，在当前项目里会驱动 GUI 里的 3D 小车运动。

- **最后一步是TTS反馈(Text_to_Speech，文字转语音)**。也就是说机器人在执行指令之后，系统会通过语音告诉用户已经执行的动作，比如“收到正在前进”等。这样系统就形成了从输入到输出到闭环。

同时，我也通过调用阿里云千问模型的 API 实现了语音智能问答系统。

**整个项目将多个模块融合成了一个稳定可运行的系统。**

#### 技术难点
        机器人听觉感知系统里有几个技术难点，我挑两个比较关键的点讲我是怎么攻克的。

**1. 自干扰抑制**

        在语音控制系统里很实际的问题就是自干扰。有时候机器人刚刚播报完一句话，如果没有自干扰抑制，麦克风就会将这句话再次采集回来，便会被系统误认为系统的新指令，直接进入自循环。

        想解决这个问题很简单。最简单的方法只需要加入延时指令，也就是在“播报后延迟一段时间再监听语音”。但是语音指令有长有短，播报长度的不同，环境的回声不同，固定长度的延时不一定能完全覆盖。

于是我做了多层保护。

**第一层是状态级保护**。只要系统处于 `TTS` 播报状态，就直接暂停监听，不再调用麦克风采集。

**第二层是时间级保护**。播报结束后，不是立刻恢复监听，而是进入一个 1.8s的冷却窗口，让尾音和环境回声先过去。1.8s基本上可以覆盖大部分回声和环境音。

**第三层是内容级保护**。我会记录最近播报过的文本，如果后续识别出的文本和最近播报内容高度相似，就认为这是回声而不是用户真实输入，然后直接忽略。

`通过三层“状态控制 + 时间窗口 + 文本相似度过滤”方案的组合保护，系统鲁棒性要比单一的延时控制更好，能满足绝大部分情况。`

**2. 声纹校验**

第二个我重点处理的难点是声纹校验，也就是 `Speaker Verification`。

在很多语音 demo 里，谁说话系统都响应，但如果这是个机器人控制系统，其实会有一个隐含问题，就是权限控制。比如周围的人随便说一句“前进”，机器人就动了，这在真实场景里其实是不合理的。

所以我引入了声纹模块。实现上，我用的是百度开源语音处理工具包 `PaddleSpeech` 里的 `ECAPA-TDNN` 模型。这是专门用于识别说话人的模型。第一次运行时，系统会引导用户录入 5 条固定语音样本，然后提取每条样本的 speaker embedding（说话人向量），建立本地声纹档案。

后续每次有新的语音输入，系统会先提取这段语音的 embedding（向量），再和已录入样本做相似度比较。我这里不是只做一次简单余弦相似度，而是综合了样本中心相似度、最近邻样本相似度和分布距离，来判断这个说话人是不是同一个人。

#### 设计取舍

**1. 为什么车控不用大模型直接生成动作**

我没有让大模型直接解析机器人控制的命令，主要因为机器人控制属于高确定性场景，它对稳定性、可解释性和低延迟要求更高。  
比如“前进1米然后左转30度”，这种命令本质上是结构化的。如果交给规则解析，其实可以非常稳定地抽取出动作类型、距离和角度，而且执行结果是可预期的。

但如果交给大模型，虽然它在语言理解上更灵活，但可能会带来更高的延迟，更高的成本。同时输出可能会出现错误，背离了高稳定性的要求。

**2. 为什么问答模式又接大模型**

因为问答模式和控制模式不一样。问答属于开放域交互，它的重点不是严格执行动作，而是理解自然语言、组织表达、保持对话流畅性。在这个场景里，大模型的优势就很明显了。

**3. 为什么做模块化设计**

我在项目里比较强调模块化，因为这个系统本质上是多能力拼接：麦克风采集、ASR、Speaker Verification、NLU、控制、TTS、GUI，这些模块未来都可能替换。

所以我在结构上把它们拆开了：

- `listener.py` 负责采集和识别
- `voiceprint.py` 负责声纹逻辑
- `config.py` 负责配置和解析规则
- `controller.py` 负责控制闭环
- `tts.py` 负责语音反馈和监听保护
- `gui_app.py` 负责 3D 展示和交互

这样拆分的好处是，如果后续我要把云端 ASR 换成本地 ASR，或者把 GUI 替换成真实机器人控制界面，不需要重写整个系统，只需要替换局部模块。

`我希望这个项目不是一次性 demo，而是具备一定演进能力的原型系统。`

#### 拓展性和落地性

1. 第三步的`ASR`部分，目前由于我本地内存和性能受限，所以我使用`SpeechRecognition`调用`Google Web Speech API`。但存在四个问题：
- 1. 强依赖网络
- 2. 中文识别效果一般
- 3. 无法本地部署
- 4. 延迟不可控

后续改进的话，我会将云端调用`Google Web Speech API`改进为本地部署`faster-whisper`，更适合部署在边缘设备上。

2. 由于我目前对整个项目进行了模块化管理，后续如果要接入真正的机器人，只需要修改`controller`模块。该模块目前默认驱动GUI的小车运动，后续可替换为真实的串口控制等。

3. 该系统目前仅加入了视觉传感，后续要提升智能性的话，还可以进行多传感器融合，比如摄像头的视觉输入，激光雷达等。

## 面试官继续追问时的展开回答

### 1. 为什么选择 `ECAPA-TDNN` 做声纹识别

因为它在说话人识别任务里是比较成熟、效果稳定的一类架构。对我这个项目来说，我更看重的是“开源可用、社区成熟、能较方便集成到 Python 项目里”，而不是自己从零训练一个声纹模型。`PaddleSpeech` 对这一块封装得比较完整，所以我选择它作为工程实现方案。

### 2. 自干扰抑制为什么要做多层保护，而不是只加一个延时

因为误触发不只是时间问题。如果只靠固定延时，短播报和长播报的效果会不一样，环境回声也不一样。有时候即使过了延时，识别内容仍然可能是刚刚播报过的话。所以我把它拆成状态控制、冷却窗口和文本相似度过滤三个层次，覆盖不同类型的误触发来源。

### 3. 连续动作命令是怎么切分和解析的

我的做法是先对文本做归一化，再根据“然后、再、接着”等连接词切分成多个 segment。每个 segment 再单独做动作类型识别，并提取距离、角度和速度修饰，最后生成结构化命令序列。这样系统就能把一句长指令拆成多个可执行动作。

### 4. 为什么控制场景更适合规则解析而不是纯 LLM

因为控制场景强调的是可预测性和安全性。规则解析虽然不如 LLM 灵活，但在动作集合固定的前提下，结果更稳定，也更容易做边界控制。大模型更适合开放式问答，而不是直接承担底层动作决策。

### 5. 如果接真实机器人，通信层准备怎么改

我会保留现有的高层控制接口，把底层动作执行从“本地模拟”替换成“真实通信”。比如把 `forward`、`turn_left` 这类结构化命令映射成 ROS 消息、串口协议或者 TCP 指令。也就是说，上层的感知和理解模块可以不动，主要改控制适配层。

### 6. 如果做成离线系统，ASR 和 TTS 分别怎么替换

ASR 可以替换为本地语音识别引擎，TTS 可以替换为离线语音合成模块。这样做的意义主要有两个：一是降低对外部网络的依赖，二是提升隐私性和部署独立性。缺点是本地模型通常对算力和部署环境要求更高，需要在效果、资源和实时性之间做平衡。